<a href="https://colab.research.google.com/github/cherifwele/test/blob/main/pyOdysseus_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **pyOdysseus : Démo web (Gradio)**

Entrées :
- 1 fichier `.txt` pivot
- plusieurs fichiers `.txt` cibles

Sorties :
- aperçu HTML dans Gradio
- fichier `.html` téléchargeable

Théoriquement, le code devrait fonctionner sur bon nombre de langues, et il les détecte automatiquement.

En ce qui concerne le modèle de détection de similarité, vous pouvez finetuner votre propre `sentence-transformer` avec le script `labse_finetuning`, lui aussi disponible sur ce dépôt : [pyOdysseus](https://github.com/OdysseusPolymetis/pyOdysseus/tree/main).

In [ ]:
!pip uninstall keras-nlp keras-hub -y
!pip install transformers==4.44.2 tokenizers==0.19.1 sentence-transformers==3.0.1 --force-reinstall -q

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
!pip -q install stanza tqdm lingua-language-detector
!pip -q install "wtpsplit[onnx-gpu]"
!pip -q install faiss-gpu-cu12 || true
!pip -q install --upgrade gradio

!rm -rf /content/pyOdysseus
!git clone -q https://github.com/OdysseusPolymetis/pyOdysseus.git /content/pyOdysseus

Quelques imports de base

In [2]:
import os
import glob
import pathlib
import subprocess
import sys
import importlib
import zipfile
from google.colab import files, drive
import torch
from lingua import LanguageDetectorBuilder
import stanza
import re
import time
import tempfile
import urllib.request

from collections import Counter
from typing import Dict, List, Tuple

from tqdm import tqdm
import gradio as gr

Fonction d'install du paquet local

In [3]:
def pip_install_editable(folder):
    p = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", folder],
        text=True,
        capture_output=True
    )
    print(p.stdout)
    print(p.stderr)
    print("returncode =", p.returncode)

In [4]:
candidates = []
for base in ["/content/pyOdysseus", "/content"]:
    for setup in glob.glob(base + "/**/setup.py", recursive=True):
        candidates.append(str(pathlib.Path(setup).parent))
    for pyproj in glob.glob(base + "/**/pyproject.toml", recursive=True):
        candidates.append(str(pathlib.Path(pyproj).parent))

priority = [p for p in candidates if "bertalign" in p.lower()]
chosen = priority[0] if priority else candidates[0]

pip_install_editable(chosen)
print("pyOdysseus Installed")

Obtaining file:///content/pyOdysseus/bertalign_odysseus
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.4 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=a5565bc000cd5be28a9938d06204252a88c06060fc045b481783e3c92b52e99e
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d

In [ ]:
PKG_ROOT = "/content/pyOdysseus/bertalign_odysseus"
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

for m in list(sys.modules):
    if m == "bertalign" or m.startswith("bertalign."):
        del sys.modules[m]

import bertalign
importlib.reload(bertalign)

print("Bertalign importé depuis :", bertalign.__file__)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_5982/975822126.py", line 9, in <cell line: 0>
    import bertalign
  File "/content/pyOdysseus/bertalign_odysseus/bertalign/__init__.py", line 9, in <module>
    from bertalign.encoder import Encoder
  File "/content/pyOdysseus/bertalign_odysseus/bertalign/encoder.py", line 2, in <module>
    from sentence_transformers import SentenceTransformer
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/__init__.py", line 10, in <module>
    from sentence_transformers.cross_encoder.CrossEncoder import CrossEncoder
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/__init__.py", line 3, in <module>
    from .CrossEncoder import CrossEncoder
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossE

# Upload du modèle finetuné (sinon LABSE par défaut).

In [ ]:
uploaded = files.upload()

Pour les deux cellules suivantes, vous devez soit choisir la première si vous avez votre modèle sur votre drive, soit la seconde si vous souhaitez uploader un modèle local.

In [ ]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/finetuned-labse.zip"
extract_root = "/content/models"
os.makedirs(extract_root, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_root)

In [ ]:
zip_name = "finetuned-labse.zip"
extract_root = "/content/models"

os.makedirs(extract_root, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_root)

print("Contenu de /content/models :", os.listdir(extract_root))

Si vous souhaitez utiliser votre propre modèle, faites tourner la cellule qui suit, sinon sautez-la.

In [ ]:
import bertalign
bertalign.set_model("/content/models/content/output/finetuned-labse", device="cuda", batch_size=32)

In [ ]:
from wtpsplit import SaT

sat = SaT("sat-3l-sm")

if torch.cuda.is_available():
    sat.half().to("cuda")
    print("SaT chargé sur GPU")
else:
    print("GPU non disponible, exécution sur CPU")

Ici les thresholds sont très variables selon les langues et les usages. J'ai mis une valeur par défaut, vous pouvez faire des tests dans les cellules qui suivent.

In [ ]:
SPLIT_THRESHOLD = 0.001

In [ ]:
sample = (
    "Voici une phrase assez longue, avec plusieurs virgules, "
    "et peut-être des coupures possibles. "
    "Et encore une autre phrase. Puis une troisième."
)

for th in [0.70, 0.20, 0.10, 0.00001]:
    segs = sat.split(sample, threshold=th)
    print(th, len(segs), segs)

Détection de la langue

In [5]:
_LANG_DETECTOR = LanguageDetectorBuilder.from_all_languages().build()
_NLP_CACHE = {}

In [6]:
def detect_lang_iso639_1(text: str, default: str = "fr") -> str:
    sample = (text or "").strip()
    if not sample:
        return default
    sample = sample[:5000]
    lang = _LANG_DETECTOR.detect_language_of(sample)
    if lang is None or lang.iso_code_639_1 is None:
        return default
    return lang.iso_code_639_1.name.lower()

Une pipeline de `stanza` optimisée pour la mémoire

In [7]:
def get_stanza_pipeline(lang: str, fallback: str = "fr"):
    lang = (lang or fallback).lower()

    if lang in _NLP_CACHE:
        return _NLP_CACHE[lang]

    def _build(l):
        stanza.download(l, verbose=False)
        return stanza.Pipeline(
            lang=l,
            processors="tokenize,lemma",
            verbose=False,
            use_gpu=torch.cuda.is_available(),
            tokenize_batch_size=64,
            pos_batch_size=256,
            lemma_batch_size=256,
        )

    try:
        _NLP_CACHE[lang] = _build(lang)
        return _NLP_CACHE[lang]
    except Exception:
        if fallback not in _NLP_CACHE:
            _NLP_CACHE[fallback] = _build(fallback)
        return _NLP_CACHE[fallback]

In [8]:
def _doc_to_token_lemmas(doc):
    tokens = []
    for sent in doc.sentences:
        for token in sent.tokens:
            for w in token.words:
                tokens.append((w.text, w.lemma))
    return tokens

In [9]:
def lemmatize_segments_batch(segments, lang: str):
    if not segments:
        return []

    nlp = get_stanza_pipeline(lang)

    try:
        docs = nlp.bulk_process(segments)
    except Exception:
        docs = [nlp(seg) for seg in segments]

    return [_doc_to_token_lemmas(doc) for doc in docs]

Test de l'import du bertalign personnalisé

In [ ]:
from bertalign import Bertalign
print("Import OK", Bertalign)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_5982/3289566085.py", line 1, in <cell line: 0>
    from bertalign import Bertalign
  File "/content/pyOdysseus/bertalign_odysseus/bertalign/__init__.py", line 9, in <module>
    from bertalign.encoder import Encoder
  File "/content/pyOdysseus/bertalign_odysseus/bertalign/encoder.py", line 2, in <module>
    from sentence_transformers import SentenceTransformer
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/__init__.py", line 10, in <module>
    from sentence_transformers.cross_encoder.CrossEncoder import CrossEncoder
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/__init__.py", line 3, in <module>
    from .CrossEncoder import CrossEncoder
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/cros

Split par défaut des textes : si vous n'avez pas "Chant", il prendra par défaut le texte en entier.

In [10]:
def split_chants(text: str):
    pattern = re.compile(r"^(Chant\s*\d+)", re.IGNORECASE)
    chants = []
    current = []

    for line in text.splitlines():
        s = line.strip()
        if pattern.match(s):
            if current:
                chants.append("\n".join(current))
                current = []
        current.append(s)

    if current:
        chants.append("\n".join(current))

    return chants if chants else [text]

In [11]:
def preprocess_chants(text_ids, src_folder, chants_folder):
    os.makedirs(chants_folder, exist_ok=True)

    for text_id in tqdm(text_ids, desc="Découpage en chants", unit="texte"):
        path = os.path.join(src_folder, f"{text_id}.txt")
        if not os.path.exists(path):
            print(f"[WARN] Introuvable: {path}")
            continue

        content = pathlib.Path(path).read_text(encoding="utf-8", errors="ignore")
        chants = split_chants(content)

        for i, chant in enumerate(chants, start=1):
            out = os.path.join(chants_folder, f"{text_id}_Chant{i}.txt")
            pathlib.Path(out).write_text(chant, encoding="utf-8")

Des mots outils personnalisés pour le français (mais c'est optionnel)

In [12]:
stopwords_url = "https://raw.githubusercontent.com/OdysseusPolymetis/colabs_for_nlp/refs/heads/main/stopwords_fr.txt"
stopwords_path = "/content/stopwords_fr.txt"
urllib.request.urlretrieve(stopwords_url, stopwords_path)

STOPWORDS = set(
    pathlib.Path(stopwords_path).read_text(encoding="utf-8", errors="ignore").splitlines()
)

print(f"{len(STOPWORDS)} stopwords chargés")

690 stopwords chargés


Fonctions d'alignement (et regroupement des index)

In [13]:
def align_pairwise(src_sents, tgt_sents):
    src_text = "\n".join(src_sents)
    tgt_text = "\n".join(tgt_sents)
    aligner = Bertalign(src_text, tgt_text, is_split=True)
    return aligner.align_sents()

In [14]:
def beads_to_cells(beads, nb_src: int):
    cells = [None] * nb_src
    pending = []
    last_anchor = None

    def attach(anchor, tgts):
        if anchor is None:
            return

        if cells[anchor] in (None, 0):
            cells[anchor] = {"rowspan": 1, "idxs": []}

        seen = set(cells[anchor]["idxs"])
        for t in tgts:
            t = int(t)
            if t not in seen:
                cells[anchor]["idxs"].append(t)
                seen.add(t)

    for sr, tr in beads:
        sr = [int(x) for x in sr]
        tr = [int(x) for x in tr]

        if not sr and tr:
            pending.extend(tr)
            continue

        if sr:
            top = sr[0]
            bot = sr[-1]
            rowspan = max(1, bot - top + 1)

            if pending:
                tr = pending + tr
                pending = []

            if cells[top] in (None, 0):
                cells[top] = {"rowspan": rowspan, "idxs": []}
            else:
                cells[top]["rowspan"] = max(cells[top]["rowspan"], rowspan)

            attach(top, tr)

            for r in range(top + 1, min(nb_src, top + rowspan)):
                cells[r] = 0

            last_anchor = top

    if pending:
        anchor = last_anchor if last_anchor is not None else (nb_src - 1 if nb_src else None)
        attach(anchor, pending)

    return cells

In [15]:
def build_cells_alignment(
    pivot_id: str,
    text_ids: List[str],
    chant_number: int,
    all_sents: Dict[str, List[str]],
    max_align: int = 5,
    top_k: int = 3,
    win: int = 5,
    skip: float = -0.1,
    margin: bool = True,
    len_penalty: bool = True,
) -> Tuple[List[str], Dict[str, List[object]]]:

    if pivot_id not in all_sents or not all_sents[pivot_id]:
        raise ValueError(f"Pivot manquant/vide dans all_sents: {pivot_id} (chant {chant_number}).")

    pivot_sents = all_sents[pivot_id]
    nb_pivot = len(pivot_sents)

    other_ids = [tid for tid in text_ids if tid != pivot_id and tid in all_sents and all_sents[tid]]
    cells_by_tid: Dict[str, List[object]] = {}

    for tid in tqdm(other_ids, desc=f"Align pivot→others (Chant {chant_number})", unit="texte"):
        tgt_sents = all_sents[tid]

        aligner = Bertalign(
            pivot_sents,
            tgt_sents,
            max_align=max_align,
            top_k=top_k,
            win=win,
            skip=skip,
            margin=margin,
            len_penalty=len_penalty,
            is_split=True,
        ).align_sents()

        beads = getattr(aligner, "result", [])
        cells_by_tid[tid] = beads_to_cells(beads, nb_src=nb_pivot)

    return pivot_sents, cells_by_tid

Calcul des fréquences (cette fonction n'est pas encore pleinement implémentée, je dois travailler dessus)

In [16]:
def new_get_freq_class(lemma, local_counter, lemma_authors, n_authors):
    if lemma in STOPWORDS:
        return "freq0"
    if local_counter.get(lemma, 0) == 1:
        return "freq1"

    authors_count = len(lemma_authors.get(lemma, []))

    if authors_count < n_authors / 4:
        return "freq2"
    if authors_count >= 0.7 * n_authors:
        if authors_count >= n_authors - 2:
            return "freq5"
        return "freq4"
    return "freq3"

In [17]:
def compute_local_lemma_freq_and_authors(row, all_tokens):
    freq = Counter()
    lemma_authors = {}

    for text_id, indices in row.items():
        if isinstance(indices, int):
            indices = [indices]
        if not indices:
            continue

        if text_id not in all_tokens:
            continue

        for idx in indices:
            if idx >= len(all_tokens[text_id]):
                continue

            for forme, lemma in all_tokens[text_id][idx]:
                freq[lemma] += 1
                lemma_authors.setdefault(lemma, set()).add(text_id)

    return freq, lemma_authors

Fonctions de render pour l'HTML

In [18]:
def render_segment(text_id, idx, all_sents, local_counter, n_authors, lemma_authors, all_tokens):
    if text_id not in all_tokens or idx >= len(all_tokens[text_id]):
        return f"<span style='color:red;'>(Segment {idx} introuvable pour {text_id})</span>"

    tokens = all_tokens[text_id][idx]
    out = []

    for forme, lemma in tokens:
        cls = new_get_freq_class(lemma, local_counter, lemma_authors, n_authors)
        tooltip = ""
        if cls != "freq0" and local_counter.get(lemma, 0) <= 1 and lemma in lemma_authors:
            tooltip = " title='Found in: " + ", ".join(sorted(lemma_authors[lemma])) + "'"
        out.append(f'<mark class="{cls}"{tooltip}>{forme}</mark>')

    return " ".join(out)

Mini CSS, vous pouvez modifier à votre guise

In [19]:
CSS_MINI = """
<style>
body { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; margin: 16px; }
h1,h2 { margin: 0.6rem 0; }
.small { color:#666; font-size: 0.9rem; }
table { width:100%; border-collapse: collapse; table-layout: fixed; }
th, td { border: 1px solid #ddd; vertical-align: top; padding: 8px; word-wrap: break-word; }
th { position: sticky; top: 0; background: #fafafa; z-index: 1; }
.blockno { width: 70px; text-align: right; color:#555; }
mark { padding: 0 2px; border-radius: 3px; }
.freq0 { background: transparent; color: #111; }
.freq1 { background: #ffd6d6; }
.freq2 { background: #ff9a9a; }
.freq3 { background: #f3f3f3; }
.freq4 { background: #d6f0ff; }
.freq5 { background: #c7ffc7; }
</style>
"""

In [21]:
def precompute_row_stats(pivot_id, text_ids, pivot_sents, cells_by_tid, all_tokens):
    other_ids = [tid for tid in text_ids if tid != pivot_id]
    row_stats = []

    for i in range(len(pivot_sents)):
        row_like = {pivot_id: i}

        for tid in other_ids:
            col = cells_by_tid.get(tid, [None] * len(pivot_sents))
            cell = col[i] if i < len(col) else None

            if isinstance(cell, dict):
                row_like[tid] = cell.get("idxs", [])
            else:
                row_like[tid] = []

        row_stats.append(compute_local_lemma_freq_and_authors(row_like, all_tokens))

    return row_stats

HTML de base

In [22]:
def generate_html_cells_only(
    pivot_id,
    text_ids,
    chant_number,
    pivot_sents,
    cells_by_tid,
    all_sents,
    all_tokens=None,
    highlight_pivot=True,
):
    n_authors = len(text_ids)
    ordered_ids = [pivot_id] + [tid for tid in text_ids if tid != pivot_id]
    other_ids = [tid for tid in ordered_ids if tid != pivot_id]

    cols = "".join([f"<th>{tid}</th>" for tid in ordered_ids])
    rows_html = []

    row_stats = precompute_row_stats(
        pivot_id=pivot_id,
        text_ids=ordered_ids,
        pivot_sents=pivot_sents,
        cells_by_tid=cells_by_tid,
        all_tokens=all_tokens,
    )

    for i in range(len(pivot_sents)):
        local_counter, local_lemma_authors = row_stats[i]
        tds = []

        if highlight_pivot:
            pivot_html = render_segment(
                pivot_id,
                i,
                all_sents,
                local_counter,
                n_authors,
                local_lemma_authors,
                all_tokens,
            )
            tds.append(f"<td>{pivot_html}</td>")
        else:
            tds.append(f"<td>{pivot_sents[i]}</td>")

        for tid in other_ids:
            col = cells_by_tid.get(tid, [None] * len(pivot_sents))
            cell = col[i] if i < len(col) else None

            if cell == 0:
                continue

            if cell is None:
                tds.append("<td>∅</td>")
                continue

            idxs = [int(x) for x in cell.get("idxs", [])]
            rowspan = int(cell.get("rowspan", 1))

            if rowspan > 1:
                src_tag = f"[{i+1}-{i+rowspan}]"
            else:
                src_tag = f"[{i+1}]"

            if not idxs:
                content = "∅"
            else:
                segs = []
                for j in idxs:
                    if tid not in all_sents or j < 0 or j >= len(all_sents[tid]):
                        continue

                    seg_html = render_segment(
                        tid,
                        j,
                        all_sents,
                        local_counter,
                        n_authors,
                        local_lemma_authors,
                        all_tokens,
                    )
                    segs.append(f"<span class='srcidx'>{src_tag}</span> {seg_html}")

                content = "<br/>".join(segs) if segs else "∅"

            rs = f' rowspan="{rowspan}"' if rowspan > 1 else ""
            tds.append(f"<td{rs}>{content}</td>")

        rows_html.append(
            "<tr>"
            + f"<td class='blockno'><b>{i+1}</b></td>"
            + "".join(tds)
            + "</tr>"
        )

    html = f"""
    <html>
      <head>
        <meta charset="utf-8">
        <title>Alignement Chant {chant_number}</title>
        {CSS_MINI}
        <style>
          .srcidx {{
            color:#666;
            font-size:0.85em;
            font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
            margin-right: 6px;
            white-space: nowrap;
          }}
        </style>
      </head>
      <body>
        <h1>Alignement — Chant {chant_number}</h1>
        <div class="small">Pivot: <b>{pivot_id}</b> • Textes: {", ".join(ordered_ids)}</div>

        <table>
          <thead>
            <tr><th class="blockno">#</th>{cols}</tr>
          </thead>
          <tbody>
            {''.join(rows_html)}
          </tbody>
        </table>
      </body>
    </html>
    """
    return html

Nettoyage basique

In [23]:
_PUNCT_ONLY = re.compile(r"^[\s\W_]+$")

def split_units_clean(text: str):
    segs = sat.split(text, threshold=SPLIT_THRESHOLD)
    segs = [s.strip() for s in segs]
    segs = [s for s in segs if s]

    merged = []
    for s in segs:
        if merged and (len(s) <= 3 or _PUNCT_ONLY.match(s)):
            merged[-1] = (merged[-1] + " " + s).strip()
        else:
            merged.append(s)

    return merged

Fonction (non présente dans le gradio, je dois l'ajouter) pour importer des zip.

In [24]:
def extract_zip_to_dir(zip_path: str, dst_dir: str):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dst_dir)

In [25]:
def infer_text_id_from_filename(path: str) -> str:
    return pathlib.Path(path).stem.strip()

Configuration de l'app Gradio.

In [26]:
def run_app(source_file, corpus_files):
    if source_file is None:
        raise gr.Error("Il faut fournir le fichier source .txt (pivot).")
    if not corpus_files:
        raise gr.Error("Il faut fournir au moins un fichier cible .txt.")

    with tempfile.TemporaryDirectory() as tmp:
        t_global = time.perf_counter()

        tmp = pathlib.Path(tmp)
        src_dir = tmp / "source"
        chants_dir = tmp / "chants"
        src_dir.mkdir(parents=True, exist_ok=True)
        chants_dir.mkdir(parents=True, exist_ok=True)

        pivot_id = infer_text_id_from_filename(source_file.name)
        pivot_path = src_dir / f"{pivot_id}.txt"
        pivot_path.write_bytes(pathlib.Path(source_file.name).read_bytes())

        for f in corpus_files:
            dst = src_dir / pathlib.Path(f.name).name
            dst.write_bytes(pathlib.Path(f.name).read_bytes())

        txt_files = sorted([p for p in src_dir.rglob("*.txt")])
        text_ids = []
        for p in txt_files:
            tid = infer_text_id_from_filename(str(p))
            if tid not in text_ids:
                text_ids.append(tid)

        if pivot_id not in text_ids:
            text_ids.insert(0, pivot_id)

        t0 = time.perf_counter()
        preprocess_chants(text_ids, str(src_dir), str(chants_dir))
        print(f"preprocess_chants: {time.perf_counter() - t0:.2f}s")

        pivot_chants = sorted(glob.glob(str(chants_dir / f"{pivot_id}_Chant*.txt")))
        if not pivot_chants:
            raise gr.Error("Aucun chant détecté pour le pivot. Vérifie le contenu / format du .txt.")

        chant_numbers = sorted(
            int(re.search(r"Chant(\d+)", pathlib.Path(f).name).group(1))
            for f in pivot_chants
        )

        html_parts = []

        for chant_number in chant_numbers:
            print(f"\n===== Chant {chant_number} =====")

            all_sents = {}
            all_langs = {}

            t0 = time.perf_counter()
            for tid in text_ids:
                p = chants_dir / f"{tid}_Chant{chant_number}.txt"
                if not p.exists():
                    continue

                content = p.read_text(encoding="utf-8", errors="ignore")
                all_langs[tid] = detect_lang_iso639_1(content, default="fr")
                all_sents[tid] = split_units_clean(content)

            print(f"split + langue: {time.perf_counter() - t0:.2f}s")

            t0 = time.perf_counter()
            all_tokens = {}
            for tid, segs in all_sents.items():
                lang = all_langs.get(tid, "fr")
                all_tokens[tid] = lemmatize_segments_batch(segs, lang)
            print(f"lemmatisation batch: {time.perf_counter() - t0:.2f}s")

            t0 = time.perf_counter()
            pivot_sents, cells_by_tid = build_cells_alignment(
                pivot_id=pivot_id,
                text_ids=text_ids,
                chant_number=chant_number,
                all_sents=all_sents,
            )
            print(f"alignement: {time.perf_counter() - t0:.2f}s")

            t0 = time.perf_counter()
            html_parts.append(
                generate_html_cells_only(
                    pivot_id=pivot_id,
                    text_ids=text_ids,
                    chant_number=chant_number,
                    pivot_sents=pivot_sents,
                    cells_by_tid=cells_by_tid,
                    all_sents=all_sents,
                    all_tokens=all_tokens,
                )
            )
            print(f"html: {time.perf_counter() - t0:.2f}s")

        final_html = "\n<hr/>\n".join(html_parts)

        out_path = pathlib.Path("/content/resultat_alignement.html")
        out_path.write_text(final_html, encoding="utf-8")

        print(f"\nTemps total: {time.perf_counter() - t_global:.2f}s")

    return final_html, str(out_path)

Appel de Gradio

In [ ]:
with gr.Blocks(title="pyOdysseus — Alignement → HTML") as demo:
    gr.Markdown(
        "# Alignement → HTML (démo)\n"
        "Pivot (.txt) + corpus (.txt multiples) → rendu HTML"
    )

    with gr.Row():
        source = gr.File(
            label="Entrée 1 — Pivot / Source (.txt)",
            file_types=[".txt"]
        )
        corpus_files = gr.Files(
            label="Entrée 2 — Textes cibles (.txt, multi-fichiers)",
            file_types=[".txt"]
        )

    btn = gr.Button("Générer")
    html_out = gr.HTML(label="Aperçu HTML")
    file_out = gr.File(label="Télécharger le .html")

    btn.click(run_app, inputs=[source, corpus_files], outputs=[html_out, file_out])

app = demo.queue(max_size=20).launch(
    share=True,
    debug=True,
    prevent_thread_lock=True,
)

print("Gradio lancé")

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95096fb6c81bffcba2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Découpage en chants: 100%|██████████| 2/2 [00:00<00:00, 153.68texte/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    retur

preprocess_chants: 0.02s

===== Chant 1 =====
